In [4]:
import os
import re
import json
import math
import time
import torch
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from groq import Groq
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
from pprint import pprint

from langchain_community.document_loaders import PyPDFLoader

d:\AIT_NLP\NLP_A6\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Task 1: Source Discovery & Data Preparation

Assigned chapter: Chapter 8  
Topic: Transformers

In [5]:
pdf_path = "D:\\AIT_NLP\\NLP_A6\\Chapter-8 RAG assignment.pdf"

loader = PyPDFLoader(pdf_path)
pages = loader.load()

print("Number of pages:", len(pages))
print("\nFirst 1000 characters from page 1:\n")
print(pages[0].page_content[:1000])

Number of pages: 27

First 1000 characters from page 1:

Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All
rights reserved. Draft of January 6, 2026.
CHAPTER
8
Transformers
“The true art of memory is the art of attention ”
Samuel Johnson, Idler #74, September 1759
In this chapter we introduce thetransformer, the standard architecture for build-
ing large language models. As we discussed in the prior chapter, transformer-based
large language models have completely changed the ﬁeld of speech and language
processing. Indeed, every subsequent chapter in this textbook will make use of them.
As with the previous chapter, we’ll focus for this chapter on the use of transformers
to model left-to-right (sometimes called causal or autoregressive) language model-
ing, in which we are given a sequence of input tokens and predict output tokens one
by one by conditioning on the prior context.
Layer Norm
+
Layer Norm
MultiHead
Attention
Feedforward
Softmax
Positi

Combining All the texts

In [6]:
full_text = "\n".join([page.page_content for page in pages])

print("Total characters:", len(full_text))
print(full_text[:1500])

Total characters: 76773
Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All
rights reserved. Draft of January 6, 2026.
CHAPTER
8
Transformers
“The true art of memory is the art of attention ”
Samuel Johnson, Idler #74, September 1759
In this chapter we introduce thetransformer, the standard architecture for build-
ing large language models. As we discussed in the prior chapter, transformer-based
large language models have completely changed the ﬁeld of speech and language
processing. Indeed, every subsequent chapter in this textbook will make use of them.
As with the previous chapter, we’ll focus for this chapter on the use of transformers
to model left-to-right (sometimes called causal or autoregressive) language model-
ing, in which we are given a sequence of input tokens and predict output tokens one
by one by conditioning on the prior context.
Layer Norm
+
Layer Norm
MultiHead
Attention
Feedforward
Softmax
Positional
Unembedding
Embedding
+
resi

Cleaning text

In [7]:
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = text.replace("\t", " ")
    text = re.sub(r" +", " ", text)
    text = re.sub(r"\n{2,}", "\n\n", text)
    return text.strip()

cleaned_text = clean_text(full_text)

print("Cleaned text length:", len(cleaned_text))
print(cleaned_text[:1500])

Cleaned text length: 76763
Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All
rights reserved. Draft of January 6, 2026.
CHAPTER
8
Transformers
“The true art of memory is the art of attention ”
Samuel Johnson, Idler #74, September 1759
In this chapter we introduce thetransformer, the standard architecture for build-
ing large language models. As we discussed in the prior chapter, transformer-based
large language models have completely changed the ﬁeld of speech and language
processing. Indeed, every subsequent chapter in this textbook will make use of them.
As with the previous chapter, we’ll focus for this chapter on the use of transformers
to model left-to-right (sometimes called causal or autoregressive) language model-
ing, in which we are given a sequence of input tokens and predict output tokens one
by one by conditioning on the prior context.
Layer Norm
+
Layer Norm
MultiHead
Attention
Feedforward
Softmax
Positional
Unembedding
Embedding
+
r

Saving Cleaned Text

In [8]:
output_dir = Path("data")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "chapter8_cleaned.txt", "w", encoding="utf-8") as f:
    f.write(cleaned_text)

print("Saved cleaned text to", output_dir / "chapter8_cleaned.txt")

Saved cleaned text to data\chapter8_cleaned.txt


In [9]:
qa_pairs = [
    {
        "question": "What is the primary purpose of the self-attention mechanism in a transformer?",
        "ground_truth_answer": "Self-attention allows a model to build contextual representations of a token by integrating information from other tokens in the sequence.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "In the attention mechanism, what roles do the query, key, and value vectors play?",
        "ground_truth_answer": "The query represents the current token being compared, the key represents tokens used for similarity comparison, and the value contains the information that is weighted and combined in the output.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "Why is a scaling factor used in the dot product of query and key vectors?",
        "ground_truth_answer": "The dot product is scaled by the square root of the key dimension to prevent large values that could cause unstable gradients during training.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "Why do transformers use multi-head attention instead of a single attention head?",
        "ground_truth_answer": "Multi-head attention allows the model to attend to different types of relationships in the sequence simultaneously using multiple attention heads.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "What components are included in a standard transformer block?",
        "ground_truth_answer": "A transformer block includes a multi-head self-attention layer, a feedforward network, residual connections, and layer normalization.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },

    {
        "question": "What is the residual stream in a transformer block?",
        "ground_truth_answer": "The residual stream is the pathway where token representations are passed through layers while each component reads from and adds its output back to the stream.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "How are positional embeddings combined with token embeddings in a transformer?",
        "ground_truth_answer": "Positional embeddings are added to token embeddings so the model can represent both the token identity and its position in the sequence.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "What is the architecture of the feedforward layer in a transformer block?",
        "ground_truth_answer": "The feedforward layer is a two-layer fully connected network applied independently to each token representation.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "What is the purpose of layer normalization in transformers?",
        "ground_truth_answer": "Layer normalization stabilizes training by normalizing activations so they have a consistent scale across the network.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "Why is masking used in the self-attention of causal language models?",
        "ground_truth_answer": "Masking prevents tokens from attending to future tokens so the model only uses previous context when predicting the next token.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "What components make up the language modeling head?",
        "ground_truth_answer": "The language modeling head consists of a linear projection called the unembedding layer followed by a softmax to produce probabilities over the vocabulary.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "What is weight tying in transformer language models?",
        "ground_truth_answer": "Weight tying refers to sharing the same weight matrix between the token embedding layer and the final unembedding layer.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "How does top-k sampling work during text generation?",
        "ground_truth_answer": "Top-k sampling restricts the probability distribution to the k most likely tokens and randomly samples the next token from that subset.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "What is the intuition behind top-p or nucleus sampling?",
        "ground_truth_answer": "Top-p sampling selects the smallest set of tokens whose cumulative probability exceeds a threshold p and samples from that set.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "What training objective is commonly used for large language models?",
        "ground_truth_answer": "Large language models are typically trained using cross-entropy loss to maximize the probability of the correct next token.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "How does the KV cache improve inference efficiency?",
        "ground_truth_answer": "The KV cache stores key and value vectors from previous tokens so they do not need to be recomputed during autoregressive generation.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "Why is attention sometimes called a token-mixing component?",
        "ground_truth_answer": "Attention is called token-mixing because it integrates information from other tokens into the representation of the current token.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "What is a decoder-only transformer model?",
        "ground_truth_answer": "A decoder-only transformer is a unidirectional model that predicts tokens autoregressively using only the decoder architecture.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "What is a limitation of absolute positional embeddings?",
        "ground_truth_answer": "Absolute positional embeddings may generalize poorly to positions near the maximum sequence length because those positions appear less frequently in training.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "What is the purpose of the logit lens tool?",
        "ground_truth_answer": "The logit lens is an interpretability method that applies the final unembedding layer to intermediate activations to analyze what the model is predicting at different layers.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    }
]

In [10]:
with open(output_dir / "qa_pairs.json", "w", encoding="utf-8") as f:
    json.dump(qa_pairs, f, indent=2, ensure_ascii=False)

print("Saved QA pairs to", output_dir / "qa_pairs_task1.json")

Saved QA pairs to data\qa_pairs_task1.json


chunking

In [11]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    
    return chunks

chunks = chunk_text(cleaned_text, chunk_size=500, overlap=50)

print("Number of chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0][:800])

Number of chunks: 171

First chunk:

Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All
rights reserved. Draft of January 6, 2026.
CHAPTER
8
Transformers
“The true art of memory is the art of attention ”
Samuel Johnson, Idler #74, September 1759
In this chapter we introduce thetransformer, the standard architecture for build-
ing large language models. As we discussed in the prior chapter, transformer-based
large language models have completely changed the ﬁeld of speech and language
processing


embedding model

In [12]:
embedding_model_name = "BAAI/bge-small-en-v1.5"

embed_tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
embed_model = AutoModel.from_pretrained(embedding_model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = embed_model.to(device)
embed_model.eval()

print("Embedding model loaded on:", device)

d:\AIT_NLP\NLP_A6\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5105.03it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key

Embedding model loaded on: cpu


In [13]:
def get_embedding(text):
    inputs = embed_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = embed_model(**inputs)
    
    # CLS token embedding
    embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
    return embedding

In [16]:
VECTOR_DB = []

for chunk in tqdm(chunks, desc="Embedding chunks"):
    emb = get_embedding(chunk)
    VECTOR_DB.append((chunk, emb))

print("Vector DB size:", len(VECTOR_DB))
print("Embedding dimension:", VECTOR_DB[0][1].shape)

Embedding chunks: 100%|██████████| 171/171 [00:12<00:00, 13.41it/s]

Vector DB size: 171
Embedding dimension: (384,)


cosine similarity + retrieval

In [14]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve(query, vector_db, top_k=3):
    query_emb = get_embedding(query)
    
    scored_chunks = []
    for chunk, emb in vector_db:
        score = cosine_similarity(query_emb, emb)
        scored_chunks.append((chunk, score))
    
    scored_chunks = sorted(scored_chunks, key=lambda x: x[1], reverse=True)
    return scored_chunks[:top_k]

test retrieval

In [17]:
test_question = "What is self-attention in a transformer?"
retrieved = retrieve(test_question, VECTOR_DB, top_k=3)

print("Question:", test_question)
print()

for i, (chunk, score) in enumerate(retrieved, 1):
    print(f"--- Retrieved chunk {i} | score={score:.4f} ---")
    print(chunk[:1000])
    print()

Question: What is self-attention in a transformer?

--- Retrieved chunk 1 | score=0.8002 ---
ding and all the outputs from all
the previous layers and blocks.
The core intuition of the transformer, and the component that distinguishes it
from the feedforward layers we saw in Chapter 6, is this multi-head attention layer,
also called a self-attention layer. Attention can be thought of as a way to build
contextual representations of a token’s meaning by attending to and integrating
information from surrounding tokens, helping the model learn how tokens relate to
each other over large span

--- Retrieved chunk 2 | score=0.7966 ---
ther linear projection WO∈ RAdv×d to reshape it, resulting in the multi-head
attention vector ai with the correct output shape [1× d] at each input i.
8.2 Transformer Blocks
The self-attention calculation lies at the core of what’s called a transformer block,
which, in addition to the self-attention layer, includes three other kinds of layers: (1)
a feedforward 

In [18]:

load_dotenv(dotenv_path=Path.cwd() / ".env")

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("Please set GROQ_API_KEY in your .env file or environment.")

client = Groq(api_key=groq_api_key)
print("Groq client ready.")

Groq client ready.


answer generation function

In [20]:
def answer_question_naive(query, vector_db, top_k=3, model_name="llama-3.1-8b-instant"):
    retrieved = retrieve(query, vector_db, top_k=top_k)
    context = "\n\n".join([chunk for chunk, _ in retrieved])

    prompt = f"""
You are a helpful assistant answering questions strictly based on the provided context.

Context:
{context}

Question:
{query}

Instructions:
- Answer using only the provided context.
- If the answer is not in the context, say: "The answer is not found in the provided context."
- Keep the answer concise, around 1-3 sentences.
"""

    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    
    answer = response.choices[0].message.content.strip()
    return answer, retrieved

manual test of full naive RAG

In [21]:
question = "What is self-attention in a transformer?"
answer, retrieved = answer_question_naive(question, VECTOR_DB, top_k=3)

print("Question:", question)
print("\nAnswer:\n", answer)
print("\nRetrieved source chunks:\n")

for i, (chunk, score) in enumerate(retrieved, 1):
    print(f"--- Chunk {i} | score={score:.4f} ---")
    print(chunk[:1000])
    print()

Question: What is self-attention in a transformer?

Answer:
 Self-attention in a transformer can be thought of as a way to build contextual representations of a token's meaning by attending to and integrating information from surrounding tokens, helping the model learn how tokens relate to each other over large spans.

Retrieved source chunks:

--- Chunk 1 | score=0.8002 ---
ding and all the outputs from all
the previous layers and blocks.
The core intuition of the transformer, and the component that distinguishes it
from the feedforward layers we saw in Chapter 6, is this multi-head attention layer,
also called a self-attention layer. Attention can be thought of as a way to build
contextual representations of a token’s meaning by attending to and integrating
information from surrounding tokens, helping the model learn how tokens relate to
each other over large span

--- Chunk 2 | score=0.7966 ---
ther linear projection WO∈ RAdv×d to reshape it, resulting in the multi-head
attention ve

20 questions through naive RAG

In [22]:
with open("data/qa_pairs.json", "r", encoding="utf-8") as f:
    qa_pairs = json.load(f)

for item in tqdm(qa_pairs, desc="Running Naive RAG"):
    question = item["question"]
    answer, _ = answer_question_naive(question, VECTOR_DB, top_k=3)
    item["naive_rag_answer"] = answer

print("Naive RAG answers generated.")

Running Naive RAG: 100%|██████████| 20/20 [00:45<00:00,  2.25s/it]

Naive RAG answers generated.


In [23]:
with open("data/qa_pairs_with_naive_rag.json", "w", encoding="utf-8") as f:
    json.dump(qa_pairs, f, indent=2, ensure_ascii=False)

print("Saved Naive RAG results.")

Saved Naive RAG results.
